# SVM Experiment Notebook

Stages:
1. Imports  
2. Configuration / paths  
3. Load dataset  
4. Load best parameters  
5. Train–test split  
6. Build pipeline  
7. Fit and predict  
8. Save last run and update best run  
9. Compute ROC and AUC  
10. Confusion matrix and combined summary  
11. Text overview  


In [10]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    roc_curve,
    auc,
    confusion_matrix,
)
import numpy as np

In [11]:
# Configuration / paths

DATA_PATH = "../../CSVs/dataset.csv"
PARAM_PATH = "../../MachineLearning/SVM/best_params.csv"
OUTPUT_DIR = "../../Results/SVMResults"
RANDOM_STATE = 100
TEST_SIZE = 0.30

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

last_result_csv = os.path.join(OUTPUT_DIR, "results_svm.csv")
best_result_csv = os.path.join(OUTPUT_DIR, "results_svm_best.csv")
roc_data_csv = os.path.join(OUTPUT_DIR, "roc_svm_clean.csv")
summary_csv = os.path.join(OUTPUT_DIR, "svm_summary.csv")

In [12]:
# 1) Load dataset

df = pd.read_csv(DATA_PATH)
y = df["anomaly"]
X = df.drop(columns=["anomaly", "timestamp", "channel", "label"], errors="ignore")

print("Shape of X:", X.shape)
print("Positive class ratio:", y.mean())

Shape of X: (2123, 21)
Positive class ratio: 0.20442769665567592


In [13]:
# 2) Load best parameters

p = pd.read_csv(PARAM_PATH).iloc[0].to_dict()
print("Using parameters:", p)

# Allow params to override defaults
RANDOM_STATE = int(p.get("random_state", RANDOM_STATE))
TEST_SIZE = float(p.get("test_size", TEST_SIZE))

Using parameters: {'kernel': 'linear', 'C': 1, 'class_weight': nan, 'gamma': 'scale', 'random_state': 100, 'test_size': 0.9}


In [14]:
# 3) Train–test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)

Train size: (212, 21) Test size: (1911, 21)


In [15]:
# 4) Build optimized SVM pipeline

kernel = str(p.get("kernel", "rbf"))
C = float(p.get("C", 1.0))
class_weight = None if str(p.get("class_weight", "None")) in ("None", "nan") else str(p.get("class_weight"))
gamma = p.get("gamma", "scale")
if kernel == "linear":
    gamma = "auto"

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        kernel=kernel,
        C=C,
        class_weight=class_weight,
        gamma=gamma,
        probability=True,  # Needed for ROC
        random_state=RANDOM_STATE
    ))
])

pipe

,steps,"[('scaler', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,C,1.0
,kernel,'linear'
,degree,3
,gamma,'auto'


In [16]:
# 5) Fit & predict

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

# Classification report
report_dict = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose().round(3)

report_df

,precision,recall,f1-score,support
0,0.913,0.980,0.945,1520.00
1,0.892,0.637,0.743,391.00
accuracy,0.910,0.910,0.910,0.91
macro avg,0.903,0.809,0.844,1911.00
weighted avg,0.909,0.910,0.904,1911.00


In [17]:
# 6) Save last run and update best run

report_df.to_csv(last_result_csv, index=True)
print(f"Saved last run results to {last_result_csv}")

def update_best_result(last_df, best_path):
    """Compare mean f1-score between `last_df` and the CSV at `best_path`.

    This function coerces the "f1-score" column to numeric (NaNs for non-numeric),
    handles missing columns, and avoids direct comparisons that raise TypeError.
    """
    # Ensure last_df is a DataFrame
    if not hasattr(last_df, '__getitem__'):
        print("Provided last_df is not a DataFrame-like object; saving it as-is.")
        last_df.to_csv(best_path, index=True)
        return

    if os.path.exists(best_path):
        try:
            best_df = pd.read_csv(best_path)
        except Exception as e:
            print(f"Could not read existing best file ({best_path}): {e}. Overwriting with last run.")
            last_df.to_csv(best_path, index=True)
            return

        # If both have a 'f1-score' column, coerce to numeric and compare means safely
        if "f1-score" in best_df.columns and "f1-score" in last_df.columns:
            last_f1 = pd.to_numeric(last_df["f1-score"], errors='coerce')
            best_f1 = pd.to_numeric(best_df["f1-score"], errors='coerce')

            last_mean = last_f1.mean()
            best_mean = best_f1.mean()

            # If best_mean is nan (no numeric values), prefer the new result
            if pd.isna(best_mean) and not pd.isna(last_mean):
                print("Existing best file has no numeric f1-scores — updating with last run.")
                last_df.to_csv(best_path, index=True)
                return

            # If last_mean is nan or both nan -> don't update
            if pd.isna(last_mean):
                print("Last run has no numeric f1-scores — not updating best result.")
                return

            # Safe numeric comparison
            try:
                if last_mean > best_mean:
                    print(f"New best model found! (F1 {last_mean:.3f} > {best_mean:.3f})")
                    last_df.to_csv(best_path, index=True)
                else:
                    print(f"Best model retained (F1 {best_mean:.3f} >= {last_mean:.3f})")
            except TypeError as e:
                print(f"TypeError comparing f1 means: {e}. Saving last run to be safe.")
                last_df.to_csv(best_path, index=True)
        else:
            # If f1-score column doesn't exist in best_df, overwrite it with last run
            print("No 'f1-score' column in existing best file — saving last run as best.")
            last_df.to_csv(best_path, index=True)
    else:
        # No existing best file -> save last run as best
        last_df.to_csv(best_path, index=True)

update_best_result(report_df, best_result_csv)

Saved last run results to ../../Results/SVMResults\results_svm.csv
Best model retained (F1 0.869 >= 0.869)


In [18]:
# 7) Compute ROC and AUC

y_proba = None
try:
    y_proba = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    roc_df = pd.DataFrame({"fpr": fpr, "tpr": tpr})
    roc_df.to_csv(roc_data_csv, index=False)
    print(f"Saved ROC data to {roc_data_csv} (AUC = {roc_auc:.3f})")
except Exception as e:
    print(f"[Warning] ROC data could not be generated: {e}")
    roc_auc = np.nan

roc_auc

Saved ROC data to ../../Results/SVMResults\roc_svm_clean.csv (AUC = 0.943)


0.9425848027998384

In [19]:
# 8) Confusion matrix and combined summary

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
cm_df = pd.DataFrame(cm, columns=["Pred 0", "Pred 1"], index=["True 0", "True 1"])

summary_data = {
    "dataset": ["clean"],
    "auc": [roc_auc],
    "accuracy": [report_dict["accuracy"]],
    "precision_0": [report_dict["0"]["precision"]],
    "recall_0": [report_dict["0"]["recall"]],
    "f1_0": [report_dict["0"]["f1-score"]],
    "precision_1": [report_dict["1"]["precision"]],
    "recall_1": [report_dict["1"]["recall"]],
    "f1_1": [report_dict["1"]["f1-score"]],
    "tp": [tp],
    "fp": [fp],
    "tn": [tn],
    "fn": [fn],
}
summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(summary_csv, index=False)
print(f"Saved summary (AUC + Confusion Matrix) to {summary_csv}")

cm_df

Saved summary (AUC + Confusion Matrix) to ../../Results/SVMResults\svm_summary.csv


,Pred 0,Pred 1
True 0,1490,30
True 1,142,249


In [20]:
# 9) Text overview

print("=== SVM Test Set Report ===")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", cm_df)
print(f"\nAUC: {roc_auc:.3f}")
print("\n=== All results and summaries saved successfully ===")

=== SVM Test Set Report ===
              precision    recall  f1-score   support

           0       0.91      0.98      0.95      1520
           1       0.89      0.64      0.74       391

    accuracy                           0.91      1911
   macro avg       0.90      0.81      0.84      1911
weighted avg       0.91      0.91      0.90      1911


Confusion Matrix:
         Pred 0  Pred 1
True 0    1490      30
True 1     142     249

AUC: 0.943

=== All results and summaries saved successfully ===
